The purpose of this notebook is to extract data from LabChart recordings and process it to attain spike data for a single unit, then export those data for analysis in the next notebook (main_part2).

# STEP 0: IMPORT MODULES

Run this code block once per session, to initialise and import modules.

In [ ]:
import hff_analysis
from hff_analysis import constants

animals = [id for id in constants.experiments.ANIMAL_DATA.keys() if id!='HFF04' and id!="HFF07"]
print("Animals:")
for id in animals:
    print(id)
print('')
sexes = [dictionary['sex'] for id, dictionary in constants.experiments.ANIMAL_DATA.items() if id in animals]
print(f"n: {len([x for x in sexes if x == 'M'])}M / {len([x for x in sexes if x == 'F'])}F")
weights = [dictionary['weight'] for id, dictionary in constants.experiments.ANIMAL_DATA.items() if id in animals]
print(f"Weights: {min(weights)}-{max(weights)} g")

# STEP 1: READ ADICHT FILE INTO PYTHON

This block only needs to be run once for each file being read,
unless `data_segments` needs to be changed.

## Arguments
* `filename` -- name of the file to be read (extension optional).
* `repetition` -- `int` for which repetition of its test the file is.
* `data_segments` -- which recording segment(s) from the file to read.
    Define a list to read segments at those indices, or set to `None`
    to read all segments in the file.
    * The duration in seconds of each segment which is read will be
      printed to assist in identifying relevant segment(s). If
      unnecessary segments have been read in this block, they can be
      filtered out in the next step without needing to rerun this block.

In [ ]:
# User defined variables
filename = 'hff19_pos3_nine-one'
repetition = 0


### User does not need to modify below this line ###

recordings = hff_analysis.read_adicht(filename)

## STEP 1a: LOAD FILEREADSETTINGS JSON

In [ ]:
import pprint

# User defined variables
frs_filename = 'frs_hff08-1_[0-1]_ampl_v2.4.0'


### User does not need to modify below this line ###

frs = hff_analysis.load_filereadsettings(frs_filename)
filename = frs.filename
repetition = frs.repetition
recording_number = frs.recording_number
epoch_timing_ms = frs.epoch_timing_ms
skip_superfast = frs.skip_superfast
spike_criteria = frs.spike_criteria
exclude_frequencies = frs.exclude_frequencies
exclude_amplitudes = frs.exclude_amplitudes
enforce_max_failrate = frs.enforce_max_failrate
print("FileReadSettings LOADED\n")
print(f"EPOCH TIMING: {epoch_timing_ms}")
print(f"SKIP SUPERFAST: {skip_superfast}")
print("SPIKE CRITERIA:")
pprint.PrettyPrinter().pprint(spike_criteria)
print(f"EXCLUDED FREQUENCIES: {exclude_frequencies}")
print(f"EXCLUDED AMPLITUDES: {exclude_amplitudes}")
print(f"ENFORCE MAX FAILRATE: {enforce_max_failrate}")
recording = hff_analysis.read_adicht(filename)[recording_number]
spikes, epochs = hff_analysis.spikes_info(
    recording,
    repetition,
    epoch_timing_ms,
    skip_superfast
)
(
    filtered_spikes,
    isi_result,
    spike_criteria_mech,
    spike_criteria_elec
) = hff_analysis.filter_trials(
    spikes,
    spike_criteria,
    exclude_frequencies,
    exclude_amplitudes,
    enforce_max_failrate
)
hff_analysis.plot_clusters(
    filtered_spikes,
    epochs,
    repetition,
    recording_number,
    epoch_timing_ms,
    spike_criteria_mech,
    spike_criteria_elec,
    isi_result,
    exclude_frequencies,
    exclude_amplitudes
)
spike_criteria = {
    'mechanical': vars(spike_criteria_mech),
    'electrical': vars(spike_criteria_elec)
}
max_failrate = hff_analysis.MAXIMUM_ISI_FAILRATE if enforce_max_failrate else None

## STEP 1b: UPDATE EXISTING SAVE FILES

In [ ]:
from os import walk
import re

json_path = '.\\outputs\\JSON\\file_read_settings\\'
json_filenames = [
    f for f in next(walk(json_path), (None, None, []))[2]
]

filename_pattern = re.compile(constants.regex.SAVED_FILENAME_REGEX)
input_filenames = [
    x for x in json_filenames if
    filename_pattern.search(x).group('testcode') == 'nine' # type: ignore
]
save_outputs = 'all'


### User does not need to modify below this line ###

hff_analysis.update_outputs(
    save_outputs,
    input_filenames,
    None,
    False,
    False
)

# STEP 2: DETECT SPIKES

This code block detects spikes. Run this block once to get a list of
spikes which will be filtered in Step 3. There is no need to rerun this
block after moving to Step 3, unless the initial settings were too
narrow (i.e. it is better to run this with generous spike detection
settings, then apply increasingly narrow filters in Step 3 until a unit
of interest is isolated).

## Arguments
* `recording_id` -- index of the recording segment to analyse.
    * Only one segment should be analysed at a time - do not specify a
      list or range!
    * Note that this index is relative to the data segment(s) chosen in
      the above step. (e.g. If `data_segments = [0, 2]` was used
      previously, then `recording_id = 1` should be used to inspect the
      third segment from the original file.)
* `epoch_timing_ms` -- timing window in milliseconds relative to each
    stimulus onset during which spikes may occur, as a `tuple` of
    numeric values.
    * The first value indicates start time and the second value end
      time.
    * The conversion from milliseconds to samples is done using
      `int()`,  which always rounds down non-integer floats. Since
      the expected sample rate is high, this level of imprecision
      should not be important. However, note that it may be possible
      for floating-point imprecision to cause misalignment of the
      epochs and spikes by one sample.
* `threshold_uV` -- threshold in microvolts above which spikes
  should be detected.

In [ ]:
# User defined variables
recording_number = 0
epoch_timing_ms = (0, 10)
skip_superfast = False


### User does not need to modify below this line ###

recording = recordings[recording_number]
spikes, epochs = hff_analysis.spikes_info(
    recording,
    repetition,
    epoch_timing_ms,
    skip_superfast
)

# STEP 3: FILTER AND PLOT SPIKES

This code block filters detected spikes and plots them. Run this block,
then tweak `spike_criteria` using the plots to exclude detected peaks
which are not from the target unit. Repeat as many times as necessary
before moving onto step 4.

## Arguments
* `spike_criteria` -- a dictionary containing `SpikeCriteria` objects
  for each stimulation type.
* `exclude_frequencies` -- a list of frequencies to exclude.
* `exclude_amplitudes` -- a list of amplitudes to exclude.
* `save_figures` -- a bool for controlling whether figures are saved.
  If this is enabled, existing figures at the target path will be
  overwritten.

In [ ]:
# User defined variables
spike_criteria = {
    'mechanical': {
        'latency_min_ms': None,
        'latency_max_ms': None,
        'size_min_uV': None,
        'size_max_uV': None
    },
    'electrical': {
        'latency_min_ms': None,
        'latency_max_ms': None,
        'size_min_uV': None,
        'size_max_uV': None
    }
}
exclude_frequencies = []
exclude_amplitudes = []
enforce_max_failrate = False
save_figures = True


### User does not need to modify below this line ###

(
    filtered_spikes,
    isi_result,
    spike_criteria_mech,
    spike_criteria_elec
) = hff_analysis.filter_trials(
    spikes,
    spike_criteria,
    exclude_frequencies,
    exclude_amplitudes,
    enforce_max_failrate
)
hff_analysis.plot_clusters(
    filtered_spikes,
    epochs,
    repetition,
    recording_number,
    epoch_timing_ms,
    spike_criteria_mech,
    spike_criteria_elec,
    isi_result,
    exclude_frequencies,
    exclude_amplitudes,
    save_figures
)
spike_criteria = {
    'mechanical': vars(spike_criteria_mech),
    'electrical': vars(spike_criteria_elec)
}
max_failrate = hff_analysis.MAXIMUM_ISI_FAILRATE if enforce_max_failrate else None

# STEP 4: SAVE DATA
Run this code block once after you are satisfied with the spike
detection to save your results as a JSON file. Repeat these 3 steps for
each file/recording segment that will be analysed, then move onto
`main_part2.ipynb`.

By default, if the target file already exists, the user will be asked
for manual confirmation before the file is overwritten. This behaviour
can be changed by setting `force_overwrite = True` at the start
of the code block.

In [ ]:
# User defined variables
force_overwrite = False
output_files = {
    'frs': {
        'filename': filename,
        'epoch_timing_ms': epoch_timing_ms,
        'skip_superfast': skip_superfast,
        'spike_criteria': spike_criteria,
        'exclude_frequencies': exclude_frequencies,
        'exclude_amplitudes': exclude_amplitudes,
        'enforce_max_failrate': enforce_max_failrate
    },
    'epochs': {
        'epochs': epochs,
        'exclude_frequencies': exclude_frequencies,
        'exclude_amplitudes': exclude_amplitudes
    },
    'spikes': {
        'isi_result': isi_result,
        'spike_criteria': spike_criteria,
        'max_failrate': max_failrate,
        'spikes': filtered_spikes
    }
}


### User does not need to modify below this line ###

for file_type, file_contents in output_files.items():
    hff_analysis.save_to_json(
        filename,
        repetition,
        recording_number,
        file_type,
        force_overwrite,
        **file_contents
    )